## Task 1. 白盒模型水印

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
from tqdm import tqdm

%load_ext autoreload
%autoreload 2
from Week14_utils_Question import LeNet5, evaluate_dataloader, compute_match_rate, random_index_generator

In [2]:
loss_func = torch.nn.CrossEntropyLoss()
learning_rate = 0.1
epoches = 10
batch_size = 128
LAMBDA = 1.
T = 64
embedding_size = 80 * 10

In [3]:
transform = transforms.Compose([transforms.ToTensor()])
trainset = torchvision.datasets.MNIST(root="./MNIST", train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root="./MNIST", train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=True, drop_last=True)

100%|██████████| 9.91M/9.91M [00:01<00:00, 8.52MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 224kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 1.84MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.26MB/s]


#### 训练与水印嵌入

In [4]:
target_b = torch.ones(T)
# 创建一个800行、64列的矩阵X_wm，极度稀疏（大部分为0），每列只有一个1、一个-1
X_wm = np.zeros((embedding_size, T))
rand_idx_gen = random_index_generator(embedding_size)
for col in range(T):
    X_wm[next(rand_idx_gen)][col] = 1.
    X_wm[next(rand_idx_gen)][col] = -1.
X_wm = torch.Tensor(X_wm)

lenet5 = LeNet5()
optimizer = torch.optim.SGD(lenet5.parameters(), lr=learning_rate)

for epoch in range(1, epoches + 1):
    for img, label in tqdm(train_loader):
        # TODO: 完成白盒水印的嵌入训练
        # NOTE: 你可以使用lenet5.get_weight()函数，获取模型最后一层的白盒权重矩阵
        # NOTE: 可能用到的API: torch.sigmoid, torch.nn.functional.binary_cross_entropy

        optimizer.zero_grad()
        # 主干分类任务损失
        pred = lenet5(img)
        loss_main = loss_func(pred, label)

        # 最⼤化【模型最后⼀个线性层的摊平参数和W 乘积得到的⼀维向量】跟【⽬标⽐特串】的匹配率，实现⽔印注⼊
        weight = lenet5.get_weight()
        
        # 权重与水印矩阵相乘得出水印比特特征
        b = torch.matmul(weight, X_wm)
        loss_wm = torch.nn.functional.binary_cross_entropy(torch.sigmoid(b), target_b)
        
        # 合并损失
        loss = loss_main + LAMBDA * loss_wm

        loss.backward()
        optimizer.step()

    evaluate_dataloader(lenet5, test_loader)
    mr = compute_match_rate(lenet5, X_wm, target_b)  # TODO: 记得完成Week14_utils_Question.py中的compute_match_rate()函数
    print(f"[Epoch {epoch}] Watermark Match Rate: {mr:.4f}")

100%|██████████| 78/78 [00:00<00:00, 132.31it/s, test_acc=0.959]


[Epoch 1] Watermark Match Rate: 1.0000


100%|██████████| 78/78 [00:00<00:00, 101.04it/s, test_acc=0.975]


[Epoch 2] Watermark Match Rate: 1.0000


100%|██████████| 78/78 [00:00<00:00, 138.90it/s, test_acc=0.981]


...

[Epoch 8] Watermark Match Rate: 1.0000


100%|██████████| 78/78 [00:00<00:00, 157.78it/s, test_acc=0.987]


[Epoch 9] Watermark Match Rate: 1.0000


100%|██████████| 78/78 [00:00<00:00, 123.66it/s, test_acc=0.984]

[Epoch 10] Watermark Match Rate: 1.0000


In [5]:
final_mr = compute_match_rate(lenet5, X_wm, target_b)
print("Final Watermark Match Rate:", final_mr)

Final Watermark Match Rate: 1.0


In [6]:
# 将模型与水印参数保存为一个字典
checkpoint = {
    'model_state_dict': lenet5.state_dict(),
    'X_wm': X_wm,
    'target_b': target_b
}
torch.save(checkpoint, "white_watermark_checkpoint.pth")
print("Model saved.")

Model saved.
